In [1]:
import numpy as np
import torch
import pandas as pd
from typing import Optional, Union

# 設定設備與型別
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
torch.set_default_dtype(dtype)
torch.set_default_device(device)

class RectangleSphericalVariablesChange:
    def __init__(self, do_x_sqrt:bool=True, return_r:bool=True):
        '''
        垂直坐標系與球坐標系之間的變數變換
        do_x_sqrt:bool = True
        => 是否需要幫輸入的 X 開根號後再做變數變換成極座標？ U = sqrt(X)

        return_r:bool = True 
        => to_theta 的 function 中：回傳的極座標資料是否需要包含 r，如果有包含的話第1個 column 就是 r
        => to_x 的 function 中：輸入的極座標資料是否有包含 r，如果沒有的話就在第一行插入1
        '''
        self.do_x_sqrt = do_x_sqrt
        self.return_r = return_r

    def to_theta(self, data:Union[np.ndarray, torch.Tensor, pd.DataFrame]) -> torch.Tensor:
        '''
        直角坐標系轉成球坐標系
        
        :param data: 直角坐標系資料
        :type data: Union[np.ndarray, torch.Tensor, pd.DataFrame]
        :return: 球坐標系資料
        :rtype: Tensor
        '''
        device = torch.get_default_device()
        dtype = torch.get_default_dtype()
        if isinstance(data, np.ndarray):
            X = torch.tensor(data, device=device, dtype=dtype)
        elif isinstance(data, torch.Tensor):
            X = data
        elif isinstance(data, pd.DataFrame):
            X = torch.tensor(np.array(data), device=device, dtype=dtype)

        # 如果希望做極座標轉換的 X 是原本的 X 開根號
        X = torch.sqrt(X) if self.do_x_sqrt else X

        # 所有維度平方相加開根號, 並且做成一個 Nx1 的矩陣
        r = torch.sqrt(torch.sum(torch.square(X), dim=1)).unsqueeze(1)

        # 計算 theta 的數量
        num_theta = X.shape[1] - 1

        # 計算每個 theta
        thetas = []
        for i in range(num_theta):
            theta_idx = i + 1
            theta = torch.atan2( torch.sqrt( torch.sum( torch.square(X[:,theta_idx:]), dim=1 ) ), X[:,i] ).unsqueeze(1)
            thetas.append(theta)

        thetas = torch.concat(thetas, dim=1)

        polor_data = torch.concat([r, thetas], dim=1) if self.return_r else thetas

        return polor_data
    
    def to_x(self, polor_data:torch.Tensor) -> torch.Tensor:
        '''
        球坐標系資料轉成直角坐標系資料
        
        :param polor_data: 球坐標系資料
        :type polor_data: torch.Tensor
        :return: 直角坐標系資料
        :rtype: Tensor
        '''
        device = torch.get_default_device()
        dtype = torch.get_default_dtype()
        
        if self.return_r: # 如果輸入的資料包含 r
            r = polor_data[:,:1]
            thetas = polor_data[:,1:]
        else: # 如果輸入的資料不包含 r
            r = torch.ones([polor_data.shape[0], 1], device=device)
            thetas = polor_data

        X = []
        sin_prod = torch.ones((thetas.shape[0], 1), device=device, dtype=dtype)
        for theta_idx in range(thetas.shape[1]):
            theta_i = thetas[:, theta_idx:theta_idx+1]
            cos_theta = torch.cos(theta_i)
            sin_theta = torch.sin(theta_i)
            x = r * sin_prod * cos_theta
            X.append(x)
            sin_prod = sin_prod * sin_theta # 為下一個 theta 轉 X 做準備
        
        # 計算最後一個維度
        x_n = r*sin_prod
        X.append(x_n)

        X = torch.concat(X, dim=1)

        # 如果當初的 X 有開根號的話就必須還原
        X = X**2 if self.do_x_sqrt else X

        return X
    
    @staticmethod
    def describe(data:torch.Tensor) -> pd.DataFrame:
        '''
        列出data 中每個 column 的平均數、標準差、最小值、第一二三四分位數、最大值、 column 中數值為0的比例
        :param data: 需要統計的資料
        :type data: torch.Tensor
        '''
        describe = pd.DataFrame(data.cpu().numpy()).describe()
        zero_ratio = (data == 0).float().mean(dim=0).cpu().numpy().round(3)

        describe.loc[len(describe)] = zero_ratio
        new_index = describe.index.tolist()
        new_index[-1] = "zero_ratio"
        describe.index = new_index

        return describe.round(3)

In [2]:
# # 建立 100 筆 5 維隨機資料
# data = np.random.rand(10, 5)
# data = torch.tensor(data)

In [3]:
# # 設定資料集的大小
# N = 10 # 資料集筆數
# D = 5 # 資料集維度

# # 生成稀疏資料
# # 每個位置有 20% 的機率為 1
# prob_matrix = torch.full((N, D), 0.5)
# mask = torch.bernoulli(prob_matrix)

# # 結合隨機數值
# sparse_data = torch.rand(N, D) * mask

In [4]:
dataset_path = '/workspaces/BO_EXPERIMENTS/src/datasets/sparse_dataset.pt'
sparse_data_info = torch.load(dataset_path)

In [5]:
sparse_data = sparse_data_info['X']

In [6]:
# 假側第一個 column 為 0
sparse_data[:,1] = 0

In [7]:
do_x_sqrt = True
# 極座標系轉換
polor_data = RectangleSphericalVariablesChange(do_x_sqrt=do_x_sqrt, return_r=True).to_theta(sparse_data)

# 轉換回直角坐標系
rectang_data = RectangleSphericalVariablesChange(do_x_sqrt=do_x_sqrt, return_r=True).to_x(polor_data)

# 四捨五入到小數點後第五位
round_rectang = torch.round(rectang_data * 1e5) * 1e-5
round_sparse = torch.round(sparse_data * 1e5) * 1e-5

print('經過極座標轉換後又轉回來的資料是否跟轉換前的相等:', torch.all(torch.round(round_sparse - round_rectang)==0).item())

經過極座標轉換後又轉回來的資料是否跟轉換前的相等: True


In [8]:
print('直角坐標系資料集:')
print(sparse_data.cpu().numpy().round(3))

print('轉換後球坐標系資料集:')
print(polor_data.cpu().numpy().round(3))

直角坐標系資料集:
[[0.    0.    0.    ... 0.012 0.012 0.075]
 [0.    0.    0.    ... 0.237 0.062 0.038]
 [0.002 0.    0.    ... 0.068 0.    0.   ]
 ...
 [0.001 0.    0.    ... 0.    0.    0.076]
 [0.    0.    0.01  ... 0.    0.    0.   ]
 [0.    0.    0.004 ... 0.    0.097 0.   ]]
轉換後球坐標系資料集:
[[1.    1.571 1.571 ... 1.571 1.221 1.193]
 [1.    1.571 1.571 ... 1.571 0.576 0.666]
 [1.    1.525 1.571 ... 1.571 0.    0.   ]
 ...
 [0.998 1.541 1.571 ... 1.571 1.571 1.571]
 [1.    1.571 1.571 ... 0.    0.    0.   ]
 [1.    1.571 1.571 ... 1.571 1.571 0.   ]]


In [17]:
torch.sum(torch.square(torch.sqrt(sparse_data)), axis=1)

tensor([1.0000, 1.0000, 0.9992, 0.9985, 1.0000, 0.9900, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 0.9993, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        0.9939, 0.9984, 1.0000, 1.0000, 1.0000, 0.9992, 0.9900, 0.9953, 1.0000,
        0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9958,
        0.9980, 1.0000, 1.0000, 1.0000, 1.0000, 0.9990, 1.0000, 1.0000, 1.0000,
        1.0000, 0.9947, 0.9948, 1.0000, 1.0000, 1.0000, 1.0000, 0.9996, 0.9990,
        1.0000, 1.0000, 1.0000, 1.0000, 0.9910, 0.9999, 0.9950, 1.0000, 1.0000,
        1.0000, 0.9968, 1.0000, 1.0000, 1.0000, 0.9977, 0.9920, 0.9900, 1.0000,
        0.9994, 1.0000, 1.0000, 0.9937, 1.0000, 1.0000, 1.0000, 0.9900, 1.0000,
        1.0000, 1.0000, 1.0000, 0.9980, 1.0000, 0.9994, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 0.9994, 1.0000, 1.0000, 0.9932, 0.9985, 0.9992, 0.9990,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9990, 1.0000, 0.9930, 1.0000,
        1.0000, 0.9992, 1.0000, 0.9999, 

In [9]:
# 統計輸入資料每個 column 的型態
RectangleSphericalVariablesChange.describe(sparse_data)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,46
count,1000.000,1000.0,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,...,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
mean,0.001,0.0,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.017,0.087,0.080,0.091,0.076,0.086,0.076,0.068,0.021,0.019
std,0.002,0.0,0.003,0.002,0.002,0.002,0.002,0.002,0.002,0.003,...,0.042,0.192,0.179,0.192,0.178,0.184,0.174,0.168,0.032,0.031
min,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
50%,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.001,0.001
75%,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.003,0.035,0.017,0.046,0.011,0.041,0.004,0.000,0.036,0.027
max,0.010,0.0,0.010,0.010,0.010,0.010,0.010,0.010,0.010,0.010,...,0.200,0.896,0.887,0.900,0.877,0.900,0.900,0.889,0.100,0.100
zero_ratio,0.691,1.0,0.681,0.705,0.713,0.716,0.684,0.711,0.695,0.703,...,0.629,0.703,0.715,0.703,0.711,0.685,0.732,0.750,0.344,0.374


In [10]:
# 統計球坐標資料每個 column 的型態
RectangleSphericalVariablesChange.describe(polor_data)

,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,46
count,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,...,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
mean,1.000,1.556,1.571,1.555,1.556,1.557,1.556,1.556,1.557,1.556,...,1.487,1.491,1.347,1.322,1.253,1.230,1.118,1.070,1.012,0.574
std,0.001,0.028,0.000,0.030,0.029,0.028,0.029,0.028,0.028,0.028,...,0.155,0.152,0.425,0.468,0.531,0.563,0.624,0.667,0.698,0.606
min,0.995,1.471,1.571,1.471,1.471,1.471,1.470,1.470,1.470,1.470,...,0.893,0.904,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,1.000,1.554,1.571,1.550,1.558,1.559,1.561,1.550,1.560,1.556,...,1.483,1.503,1.322,1.336,1.004,0.999,0.510,0.366,0.091,0.000
50%,1.000,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,...,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,0.352
75%,1.000,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,...,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.158
max,1.000,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,...,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571,1.571
zero_ratio,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.012,0.032,0.053,0.087,0.131,0.174,0.231,0.374


In [11]:
x = torch.tensor([[ 2/3, 0, 0, 1/3, 0 ]])
single_polor_data = RectangleSphericalVariablesChange(do_x_sqrt=True, return_r=True).to_theta(x)
print('排序前的資料轉換成球坐標')
print(single_polor_data.cpu().numpy().round(3))

x = torch.tensor([[1/3, 2/3, 0, 0, 0 ]])
single_polor_data = RectangleSphericalVariablesChange(do_x_sqrt=True, return_r=True).to_theta(x)
print('排序後的資料轉換成球坐標')
print(single_polor_data.cpu().numpy().round(3))

排序前的資料轉換成球坐標
[[1.    0.615 1.571 1.571 0.   ]]
排序後的資料轉換成球坐標
[[1.    0.955 0.    0.    0.   ]]
